In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv(r'C:\Users\apaks\Desktop\Real Estate Project\artifacts\data\preprocessed-data\gurgaon_properties_post_feature_selection.csv')

In [3]:
df.head()

,property_type,sector,price,bedRoom,bathroom,balcony,agePossession,built_up_area,servant room,store room,furnishing_type,luxury_category,floor_category
0,flat,sector 83,1.18,3,3,2,relatively new,1612.0,0,0,unfurnished,budget,high-rise
1,flat,sector 62,4.00,3,3,3,new,2353.0,1,0,semifurnished,budget,high-rise
2,flat,sector 85,1.10,2,2,3+,relatively new,1459.0,0,1,unfurnished,luxury,low-rise
3,flat,sector 91,0.89,2,2,3+,relatively new,1297.0,0,0,semifurnished,budget,medium-rise
4,flat,sector 7,0.48,3,2,1,new,889.0,0,0,unfurnished,budget,low-rise


In [5]:
sorted(df['sector'].unique())

['sector 1',
 'sector 10',
 'sector 102',
 'sector 103',
 'sector 104',
 'sector 105',
 'sector 106',
 'sector 107',
 'sector 108',
 'sector 109',
 'sector 11',
 'sector 110',
 'sector 111',
 'sector 112',
 'sector 113',
 'sector 12',
 'sector 13',
 'sector 14',
 'sector 15',
 'sector 17',
 'sector 2',
 'sector 21',
 'sector 22',
 'sector 23',
 'sector 24',
 'sector 25',
 'sector 26',
 'sector 27',
 'sector 28',
 'sector 3',
 'sector 30',
 'sector 31',
 'sector 33',
 'sector 36',
 'sector 37',
 'sector 38',
 'sector 39',
 'sector 3a',
 'sector 4',
 'sector 40',
 'sector 41',
 'sector 43',
 'sector 45',
 'sector 46',
 'sector 47',
 'sector 48',
 'sector 49',
 'sector 5',
 'sector 50',
 'sector 51',
 'sector 52',
 'sector 53',
 'sector 54',
 'sector 55',
 'sector 56',
 'sector 57',
 'sector 58',
 'sector 59',
 'sector 6',
 'sector 60',
 'sector 61',
 'sector 62',
 'sector 63',
 'sector 65',
 'sector 66',
 'sector 67',
 'sector 68',
 'sector 69',
 'sector 7',
 'sector 70',
 'sector 71',
 

In [33]:
from dotenv import load_dotenv
import os

load_dotenv()
API_KEY = os.getenv('GOOGLE_MAPS_API_KEY')

In [48]:
import requests
import json

def get_lat_long(sector, API_KEY):
    url = "https://maps.googleapis.com/maps/api/geocode/json"
    params = {'address': f'{sector} gurgaon',
              'key': API_KEY}
    r = requests.get(url, params)

    result_lat = r.json()['results'][0]['geometry']['location']['lat']
    result_lng = r.json()['results'][0]['geometry']['location']['lng']
    
    return result_lat, result_lng


In [39]:
get_lat_long('sector 21', API_KEY)

28.5133929 77.0722759


(28.5133929, 77.0722759)

In [46]:
all_sectors  = df['sector'].unique()

In [47]:
cordinates_dict = {}
for sector in all_sectors:
    cordinates_dict[sector] = get_lat_long(sector, API_KEY)

print(cordinates_dict)

28.3985667 76.9646908
28.4139095 77.08859439999999
28.40418 76.9513432
28.4013829 76.92253199999999
28.4643736 77.0142942
28.4749763 76.971507
28.3924859 77.0540907
28.4844116 77.0859978
28.3866865 76.9484624
28.4275535 77.0491679
28.407854 76.9153281
28.3965979 77.034108
28.3623546 76.97870759999999
28.4157699 77.01182460000001
28.4948731 76.9844677
28.3971448 77.08666459999999
28.4171926 76.9081238
28.5072801 77.0089452
28.4433834 77.0515376
28.5238449 77.0319784
28.4176955 77.0359426
28.4756706 77.03919669999999
28.478782 76.9959872
28.4231993 77.07521059999999
28.5095068 77.0319784
28.4433129 77.0947514
28.4103693 77.028145
28.4375165 76.9999474
28.513372 76.9830277
28.3801295 76.9844677
28.3935416 76.95998519999999
28.4710811 77.0454144
28.4909088 77.01758319999999
28.4728421 77.08646139999999
28.4225192 77.0211379
28.4603762 77.0578855
28.4244603 77.09912419999999
28.528745 77.0233415
28.4794366 77.01758319999999
28.4030424 77.06900519999999
28.4085198 76.9369384
28.4320397 77.06

In [52]:
cordinates_df = pd.DataFrame(cordinates_dict).T

In [55]:
cordinates_df = cordinates_df.reset_index().rename(columns = {'index': 'sector', 0: 'lat', 1:'lon'})

In [57]:
# merge original df with the cordinates df
df = df.merge(cordinates_df, on='sector')

In [58]:
df.to_csv(r'C:\Users\apaks\Desktop\Real Estate Project\artifacts\data\preprocessed-data\gurgaon_properties_with_lat_long.csv', index = False)

In [60]:
sector_wise_stats = df.groupby('sector').agg(
    avg_price_sqft = ('price', 'mean'),
    avg_builtup = ('built_up_area', 'mean'),
    lat = ('lat', 'first'),
    lon = ('lon', 'first')
).reset_index()

In [61]:
import branca.colormap as cm

min_price = sector_wise_stats["avg_price_sqft"].min()
max_price = sector_wise_stats["avg_price_sqft"].max()

colormap = cm.linear.YlOrRd_09.scale(min_price, max_price)

def price_to_color(price): 
    return colormap(price)


In [62]:
import folium

m = folium.Map(location=[28.4595, 77.0266], zoom_start=12)

for _, row in sector_wise_stats.iterrows():
    folium.CircleMarker(
        location=[row["lat"], row["lon"]],
        radius=row["avg_builtup"] / 200,   # scale down so bubbles aren't huge
        fill=True,
        fill_color=price_to_color(row["avg_price_sqft"]),
        color=None,
        fill_opacity=0.7,
        popup=(
            f"<b>Sector:</b> {row['sector']}<br>"
            f"<b>Avg Price/sqft:</b> ₹{row['avg_price_sqft']:.0f}<br>"
            f"<b>Avg Built-up:</b> {row['avg_builtup']:.0f} sqft"
        )
    ).add_to(m)


In [63]:
colormap.caption = "Average Price per Sqft (₹)"
colormap.add_to(m)


In [65]:
m